# 04 — Phase 2 결합 · Gate 2 (M5)

**사전 잠금 (결과 보기 전 확정 — 변경 금지)**
- 변동성 대표 = `vix` 단일 (사전등록 대표. realized/blend 성과우위로 교체 금지)
- 추세 대표 = `mean(w_ma200, w_tsmom12)` (동률 방어, 성과선택 회피 위해 등가중 평균)
- 결합 = `mean(w_vol, w_trend)` 등가중 — 신호 가중치 데이터 최적화 절대 금지
- 신용 EXCLUDE 유지 (M4 확정)

**공통기간:** 1990-01-31~ (realized·blend 워밍업 바닥, N=9148)

**대조군 3종:**
- ee: 결합 실현 평균노출 고정배분 (band=0)
- tsmom12: 최고 단일 신호 (M4 Gate 1 기준)
- buy&hold: S&P500 총수익

**Gate 2 기준:** 결합이 (a) tsmom12 AND (b) ee 둘 다를 Sharpe/Sortino/Calmar로 이기는가?

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if '' not in sys.path:
    sys.path.insert(0, os.getcwd())

import warnings, yaml
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

from src.data_loader import load_all
from src.indicators.base import standalone_data
from src.indicators.volatility import VolatilityIndicator
from src.indicators.trend import TrendIndicator
from src.combine import combine_equal_weight
from src.backtest import run
from src.benchmarks import buy_and_hold, equal_exposure
from src.metrics import summary, avg_exposure, annual_vol

with open('config/base.yaml') as f:
    cfg = yaml.safe_load(f)
data_raw = load_all(cfg)

COMMON_START = pd.Timestamp('1990-01-31')

[캐시 사용] /workspaces/Strategy_Triangulate/data/raw_prices.parquet
[캐시 사용] /workspaces/Strategy_Triangulate/data/raw_fred.parquet


## 1. 신호 계산 및 결합 구성

In [2]:
# ★ 사전 잠금: 변동성=vix, 추세=mean(ma200,tsmom12)
data_vol = standalone_data(data_raw, 'volatility')
data_trd = standalone_data(data_raw, 'trend')

w_vix_full   = VolatilityIndicator().signal(data_vol, {**cfg, 'vol_estimator': 'vix'})
w_ma200_full = TrendIndicator().signal(data_trd, {**cfg, 'rule': 'ma200',   'w_floor': 0.0})
w_tsmom_full = TrendIndicator().signal(data_trd, {**cfg, 'rule': 'tsmom12', 'w_floor': 0.0})

# ★ 공통기간 자르기 FIRST, 그 후 w_trend 구성 (NaN 섞임 방지)
w_vix   = w_vix_full.loc[COMMON_START:].dropna()
w_ma200 = w_ma200_full.loc[COMMON_START:].dropna()
w_tsmom = w_tsmom_full.loc[COMMON_START:].dropna()

assert w_vix.isna().sum()   == 0, 'w_vix NaN 존재 — 정합 오류'
assert w_ma200.isna().sum() == 0, 'w_ma200 NaN 존재 — 정합 오류'
assert w_tsmom.isna().sum() == 0, 'w_tsmom NaN 존재 — 정합 오류'

# 추세 대표: 등가중 평균 (성과선택 금지)
w_trend = (w_ma200 + w_tsmom) / 2
assert w_trend.isna().sum() == 0, 'w_trend NaN 존재 — 정합 오류'

# 결합: mean(w_vol, w_trend) — combine_equal_weight가 NaN assert 수행
w_comb = combine_equal_weight({'vol': w_vix, 'trend': w_trend}, cfg)

print(f'공통기간: {w_comb.index[0].date()} ~ {w_comb.index[-1].date()}  N={len(w_comb)}')
print(f'w_vix   평균={w_vix.mean():.4f}  std={w_vix.std():.4f}')
print(f'w_trend 평균={w_trend.mean():.4f}  std={w_trend.std():.4f}')
print(f'w_comb  평균={w_comb.mean():.4f}  std={w_comb.std():.4f}')

공통기간: 1990-01-31 ~ 2026-05-29  N=9148
w_vix   평균=0.6863  std=0.2037
w_trend 평균=0.8029  std=0.3637
w_comb  평균=0.7446  std=0.2550


## 2. 전체기간 백테스트 (net)

In [3]:
sp500tr = data_vol['sp500tr']
r_eq    = sp500tr.pct_change().reindex(w_comb.index).fillna(0.0)
rf_ann  = data_raw['rf'].reindex(w_comb.index)   # run()용: annual %
rf_d    = rf_ann / 100 / 252                       # summary()용: daily rate

res_comb  = run(w_comb,  r_eq, rf_ann, cfg)
res_tsmom = run(w_tsmom, r_eq, rf_ann, cfg)
res_vix   = run(w_vix,   r_eq, rf_ann, cfg)
res_trend = run(w_trend, r_eq, rf_ann, cfg)
bh_res    = buy_and_hold(r_eq, rf_ann, cfg)

mw_c  = avg_exposure(res_comb['weights'])
mw_t  = avg_exposure(res_tsmom['weights'])
mw_v  = avg_exposure(res_vix['weights'])
mw_tr = avg_exposure(res_trend['weights'])

ee_c  = equal_exposure(mw_c,  r_eq, rf_ann, cfg)
ee_t  = equal_exposure(mw_t,  r_eq, rf_ann, cfg)
ee_v  = equal_exposure(mw_v,  r_eq, rf_ann, cfg)
ee_tr = equal_exposure(mw_tr, r_eq, rf_ann, cfg)

m_comb  = summary(res_comb['equity_net'],  res_comb['returns_net'],  r_eq, rf_d, res_comb['weights'],  res_comb['turnover'])
m_tsmom = summary(res_tsmom['equity_net'], res_tsmom['returns_net'], r_eq, rf_d, res_tsmom['weights'], res_tsmom['turnover'])
m_vix   = summary(res_vix['equity_net'],   res_vix['returns_net'],   r_eq, rf_d, res_vix['weights'],   res_vix['turnover'])
m_trend = summary(res_trend['equity_net'], res_trend['returns_net'], r_eq, rf_d, res_trend['weights'], res_trend['turnover'])
m_ee_c  = summary(ee_c['equity_net'],  ee_c['returns_net'],  r_eq, rf_d, ee_c['weights'],  ee_c['turnover'])
m_ee_t  = summary(ee_t['equity_net'],  ee_t['returns_net'],  r_eq, rf_d, ee_t['weights'],  ee_t['turnover'])
m_bh    = summary(bh_res['equity_net'], bh_res['returns_net'], r_eq, rf_d, bh_res['weights'], bh_res['turnover'])

# ★ ee 평균노출 일치 확인
assert abs(mw_c - m_ee_c['avg_exposure']) < 1e-6, f'ee 평균노출 불일치: {mw_c:.6f} vs {m_ee_c["avg_exposure"]:.6f}'
print(f'ee 평균노출 일치 확인: combo AvgExp={mw_c:.4f} == ee AvgExp={m_ee_c["avg_exposure"]:.4f}')

hdr = f'{"전략":12s}  {"Sharpe":>7} {"Sortino":>8} {"Calmar":>7} {"CAGR":>8} {"Vol":>7} {"MDD":>8} {"UpCap":>7} {"DnCap":>7} {"Turn/yr":>8} {"AvgExp":>7}'
print()
print(hdr)
print('-' * len(hdr))
for label, m in [('combo', m_comb), ('ee(combo)', m_ee_c), ('tsmom12', m_tsmom), ('ee(tsmom)', m_ee_t),
                  ('vix', m_vix), ('trend_avg', m_trend), ('buy_hold', m_bh)]:
    print(f'{label:12s}  {m["sharpe"]:7.3f} {m["sortino"]:8.3f} {m["calmar"]:7.3f} '
          f'{m["cagr"]:7.2%} {m["annual_vol"]:6.2%} {m["mdd"]:7.2%} '
          f'{m["up_capture"]:6.2%} {m["down_capture"]:6.2%} '
          f'{m["annual_turnover"]:7.2f} {m["avg_exposure"]:6.3f}')

ee 평균노출 일치 확인: combo AvgExp=0.7405 == ee AvgExp=0.7405

전략             Sharpe  Sortino  Calmar     CAGR     Vol      MDD   UpCap   DnCap  Turn/yr  AvgExp
-------------------------------------------------------------------------------------------------
combo           0.634    0.888   0.473   9.06% 10.22% -19.16% 65.23% 66.55%    3.55  0.741
ee(combo)       0.531    0.757   0.214   9.31% 13.33% -43.46% 72.58% 75.80%    0.39  0.741
tsmom12         0.631    0.880   0.344  10.77% 13.34% -31.32% 76.71% 73.63%    2.84  0.825
ee(tsmom)       0.531    0.757   0.210   9.96% 14.86% -47.54% 81.23% 83.98%    0.30  0.825
vix             0.559    0.785   0.274   7.75%  9.27% -28.33% 58.91% 64.46%    4.30  0.684
trend_avg       0.659    0.922   0.428  10.37% 11.96% -24.26% 72.21% 69.45%    4.47  0.803
buy_hold        0.531    0.757   0.204  11.24% 18.01% -55.25% 99.76% 100.00%    0.03  1.000


## 3. 하위구간 분할 (전반 1990~2007 / 후반 2008~)

In [4]:
SPLITS = [
    ('전체',           COMMON_START,              sp500tr.index[-1]),
    ('전반_1990_2007', COMMON_START,              pd.Timestamp('2007-12-31')),
    ('후반_2008_',     pd.Timestamp('2008-01-01'), sp500tr.index[-1]),
]

all_rows = []

for period_name, ps, pe in SPLITS:
    r_eq_p  = sp500tr.pct_change().reindex(w_comb.index).fillna(0.0).loc[ps:pe]
    rf_ann_p = data_raw['rf'].reindex(w_comb.index).loc[ps:pe]
    rf_d_p   = rf_ann_p / 100 / 252

    def bt(w_p, label, family):
        res = run(w_p, r_eq_p, rf_ann_p, cfg)
        mw  = avg_exposure(res['weights'])
        ee  = equal_exposure(mw, r_eq_p, rf_ann_p, cfg)
        bh  = buy_and_hold(r_eq_p, rf_ann_p, cfg)
        m   = summary(res['equity_net'],  res['returns_net'],  r_eq_p, rf_d_p, res['weights'],  res['turnover'])
        me  = summary(ee['equity_net'],   ee['returns_net'],   r_eq_p, rf_d_p, ee['weights'],   ee['turnover'])
        mbh = summary(bh['equity_net'],   bh['returns_net'],   r_eq_p, rf_d_p, bh['weights'],   bh['turnover'])
        all_rows.append({
            'period': period_name, 'strategy': label, 'family': family,
            'period_start': str(ps.date()), 'period_end': str(pe.date()),
            'CAGR':   f'{m["cagr"]:.2%}',   'Vol':    f'{m["annual_vol"]:.2%}',
            'Sharpe': round(m['sharpe'],3),  'Sortino': round(m['sortino'],3),
            'MDD':    f'{m["mdd"]:.2%}',     'Calmar':  round(m['calmar'],3),
            'UpCap':  f'{m["up_capture"]:.2%}',  'DnCap': f'{m["down_capture"]:.2%}',
            'Turnover_yr': round(m['annual_turnover'],2),
            'AvgExp': round(m['avg_exposure'],3),
            'ee_Sharpe':  round(me['sharpe'],3),  'ee_Sortino': round(me['sortino'],3),
            'ee_Calmar':  round(me['calmar'],3),  'ee_CAGR': f'{me["cagr"]:.2%}',
            'ee_Vol':  f'{me["annual_vol"]:.2%}', 'ee_MDD': f'{me["mdd"]:.2%}',
            'bh_Sharpe': round(mbh['sharpe'],3),  'bh_CAGR': f'{mbh["cagr"]:.2%}',
        })

    bt(w_comb.loc[ps:pe],  'combo',     'combination')
    bt(w_tsmom.loc[ps:pe], 'tsmom12',   'trend')
    bt(w_vix.loc[ps:pe],   'vix',       'volatility')
    bt(w_trend.loc[ps:pe], 'trend_avg', 'trend')

    bh  = buy_and_hold(r_eq_p, rf_ann_p, cfg)
    mbh = summary(bh['equity_net'], bh['returns_net'], r_eq_p, rf_d_p, bh['weights'], bh['turnover'])
    all_rows.append({
        'period': period_name, 'strategy': 'buy_hold', 'family': 'benchmark',
        'period_start': str(ps.date()), 'period_end': str(pe.date()),
        'CAGR': f'{mbh["cagr"]:.2%}', 'Vol': f'{mbh["annual_vol"]:.2%}',
        'Sharpe': round(mbh['sharpe'],3), 'Sortino': round(mbh['sortino'],3),
        'MDD': f'{mbh["mdd"]:.2%}', 'Calmar': round(mbh['calmar'],3),
        'UpCap': f'{mbh["up_capture"]:.2%}', 'DnCap': f'{mbh["down_capture"]:.2%}',
        'Turnover_yr': round(mbh['annual_turnover'],2), 'AvgExp': round(mbh['avg_exposure'],3),
        'ee_Sharpe': '-', 'ee_Sortino': '-', 'ee_Calmar': '-',
        'ee_CAGR': '-', 'ee_Vol': '-', 'ee_MDD': '-',
        'bh_Sharpe': round(mbh['sharpe'],3), 'bh_CAGR': f'{mbh["cagr"]:.2%}',
    })

df_all = pd.DataFrame(all_rows)
df_all.to_csv('results/combo_metrics.csv', index=False)
print('저장: results/combo_metrics.csv')
print(df_all[['period','strategy','Sharpe','Sortino','Calmar','CAGR','Vol','MDD','AvgExp',
              'ee_Sharpe','ee_Sortino','ee_Calmar']].to_string(index=False))

저장: results/combo_metrics.csv
      period  strategy  Sharpe  Sortino  Calmar   CAGR    Vol     MDD  AvgExp ee_Sharpe ee_Sortino ee_Calmar
          전체     combo   0.634    0.888   0.473  9.06% 10.22% -19.16%   0.741     0.531      0.757     0.214
          전체   tsmom12   0.631    0.880   0.344 10.77% 13.34% -31.32%   0.825     0.531      0.757      0.21
          전체       vix   0.559    0.785   0.274  7.75%  9.27% -28.33%   0.684     0.531      0.758     0.218
          전체 trend_avg   0.659    0.922   0.428 10.37% 11.96% -24.26%   0.803     0.531      0.757     0.211
          전체  buy_hold   0.531    0.757   0.204 11.24% 18.01% -55.25%   1.000         -          -         -
전반_1990_2007     combo   0.569    0.820   0.517  9.78% 10.10% -18.90%   0.747     0.481      0.702     0.261
전반_1990_2007   tsmom12   0.702    1.024   0.677 12.99% 12.72% -19.19%   0.833     0.482      0.701     0.249
전반_1990_2007       vix   0.467    0.672   0.290  8.21%  9.02% -28.33%   0.692     0.481      0.703

## 4. Gate 2 판정

In [5]:
print('=' * 65)
print('Gate 2 판정: 결합 vs (a) tsmom12  AND  (b) ee  — 3지표 기준')
print('=' * 65)

gate2_overall = None

for period_name in ['전체', '전반_1990_2007', '후반_2008_']:
    c = df_all[(df_all['period'] == period_name) & (df_all['strategy'] == 'combo')].iloc[0]
    t = df_all[(df_all['period'] == period_name) & (df_all['strategy'] == 'tsmom12')].iloc[0]

    vs_ee = [
        c['Sharpe']  > c['ee_Sharpe'],
        c['Sortino'] > c['ee_Sortino'],
        c['Calmar']  > c['ee_Calmar'],
    ]
    vs_ts = [
        c['Sharpe']  > t['Sharpe'],
        c['Sortino'] > t['Sortino'],
        c['Calmar']  > t['Calmar'],
    ]
    pass_ee = sum(vs_ee) >= 2
    pass_ts = sum(vs_ts) >= 2

    print(f'\n[{period_name}]')
    print(f'  combo   Sh={c["Sharpe"]} So={c["Sortino"]} Ca={c["Calmar"]}')
    print(f'  ee      Sh={c["ee_Sharpe"]} So={c["ee_Sortino"]} Ca={c["ee_Calmar"]}')
    print(f'  tsmom12 Sh={t["Sharpe"]} So={t["Sortino"]} Ca={t["Calmar"]}')
    print(f'  vs ee:      {sum(vs_ee)}/3 {["✅" if v else "❌" for v in vs_ee]} → {"PASS" if pass_ee else "FAIL"}')
    print(f'  vs tsmom12: {sum(vs_ts)}/3 {["✅" if v else "❌" for v in vs_ts]} → {"PASS" if pass_ts else "FAIL"}')

    if period_name == '전체':
        gate2_overall = pass_ee and pass_ts
        print(f'\n  ★ Gate 2 (전체기간): {"PASS" if gate2_overall else "FAIL"}')

print()
print('─' * 65)
print(f'★ Gate 2 최종 판정: {"PASS" if gate2_overall else "FAIL"}')
print()
print('[해석]')
print('- 전체기간: combo가 ee·tsmom12 둘 다 위험조정 3지표로 상회 → PASS')
print('- 단 combo vs tsmom12 차이는 극히 미미: Sharpe 0.634 vs 0.631 (0.003 차이)')
print('- 전반(1990~2007, 조용한 강세장 포함): combo가 tsmom12에 명확 열위 (0/3)')
print('  → combo의 전체기간 우위는 후반(2008~) 구간에 의존')
print('- 후반(2008~): combo가 tsmom12를 3/3으로 상회 (GFC·2020·2022 방어 기여)')
print('- 결론: 형식적 Gate 2 PASS이나, 하위구간 의존성과 근소한 우위를 인지할 것')

Gate 2 판정: 결합 vs (a) tsmom12  AND  (b) ee  — 3지표 기준

[전체]
  combo   Sh=0.634 So=0.888 Ca=0.473
  ee      Sh=0.531 So=0.757 Ca=0.214
  tsmom12 Sh=0.631 So=0.88 Ca=0.344
  vs ee:      3/3 ['✅', '✅', '✅'] → PASS
  vs tsmom12: 3/3 ['✅', '✅', '✅'] → PASS

  ★ Gate 2 (전체기간): PASS

[전반_1990_2007]
  combo   Sh=0.569 So=0.82 Ca=0.517
  ee      Sh=0.481 So=0.702 Ca=0.261
  tsmom12 Sh=0.702 So=1.024 Ca=0.677
  vs ee:      3/3 ['✅', '✅', '✅'] → PASS
  vs tsmom12: 0/3 ['❌', '❌', '❌'] → FAIL

[후반_2008_]
  combo   Sh=0.702 So=0.959 Ca=0.478
  ee      Sh=0.579 So=0.816 Ca=0.228
  tsmom12 Sh=0.574 So=0.774 Ca=0.279
  vs ee:      3/3 ['✅', '✅', '✅'] → PASS
  vs tsmom12: 3/3 ['✅', '✅', '✅'] → PASS

─────────────────────────────────────────────────────────────────
★ Gate 2 최종 판정: PASS

[해석]
- 전체기간: combo가 ee·tsmom12 둘 다 위험조정 3지표로 상회 → PASS
- 단 combo vs tsmom12 차이는 극히 미미: Sharpe 0.634 vs 0.631 (0.003 차이)
- 전반(1990~2007, 조용한 강세장 포함): combo가 tsmom12에 명확 열위 (0/3)
  → combo의 전체기간 우위는 후반(2008~) 구간에 의존
- 후반(2008

## 5. 분산이득 진단

In [6]:
# 전략 변동성
vol_comb  = annual_vol(res_comb['returns_net'])
vol_vix   = annual_vol(res_vix['returns_net'])
vol_trend = annual_vol(res_trend['returns_net'])
vol_avg   = (vol_vix + vol_trend) / 2

# 능동수익 상관 (M4 (A) 방식: 전략 − ee)
ee_v_ret  = ee_v['returns_net'].reindex(res_vix['returns_net'].index)
ee_tr_ret = ee_tr['returns_net'].reindex(res_trend['returns_net'].index)
active_vix   = res_vix['returns_net']   - ee_v_ret
active_trend = res_trend['returns_net'] - ee_tr_ret
corr_A = active_vix.corr(active_trend)
corr_i = w_vix.corr(w_trend)

# 이론적 등가중 분산이득
theo_ratio = np.sqrt((1 + corr_A) / 2)   # vol_comb / vol_avg 이론치 (단순 등가중)
actual_ratio = vol_comb / vol_avg

print('=== 분산이득 진단 (M4 예상 검증) ===')
print()
print(f'[연율 변동성]')
print(f'  vix 전략:     {vol_vix:.4f} ({vol_vix:.2%})')
print(f'  trend_avg:    {vol_trend:.4f} ({vol_trend:.2%})')
print(f'  입력 평균:    {vol_avg:.4f} ({vol_avg:.2%})')
print(f'  combo:        {vol_comb:.4f} ({vol_comb:.2%})')
print(f'  분산이득 있음 (combo < 입력평균): {vol_comb < vol_avg}')
print(f'  축소폭: {vol_avg - vol_comb:.4f} ({(vol_avg - vol_comb)/vol_avg*100:.1f}%)')
print()
print(f'[신호 및 능동수익 상관]')
print(f'  (i) 신호상관 vix↔trend_avg: {corr_i:.4f}  [M4 회색지대 상단 근방]')
print(f'  (A) 능동수익 상관 vix↔trend: {corr_A:.4f}  [M4 기록: ~0.80 고상관]')
print()
print(f'[분산이득 이론 vs 실현]')
print(f'  이론적 비율 sqrt((1+ρ_A)/2) = sqrt((1+{corr_A:.3f})/2) = {theo_ratio:.4f}')
print(f'  실현 비율 combo_vol/avg_vol  = {vol_comb:.4f}/{vol_avg:.4f}   = {actual_ratio:.4f}')
print(f'  → 실현({actual_ratio:.4f}) > 이론({theo_ratio:.4f}): 분산이득이 이론치보다 얇음')
print()
print('[해석]')
print('M4에서 예상: "(A) 0.80 고상관 → 분산이득 약함"')
print(f'실제: 변동성 축소 {(vol_avg-vol_comb)/vol_avg*100:.1f}% (3.7%p 감소)')
print('→ 분산이득은 실재하지만 M4 예상대로 얇다.')
print('→ Gate 2 근소 우위(Sharpe +0.003)는 분산이득보다 개별 신호 조합 효과가 큼.')

=== 분산이득 진단 (M4 예상 검증) ===

[연율 변동성]
  vix 전략:     0.0927 (9.27%)
  trend_avg:    0.1196 (11.96%)
  입력 평균:    0.1061 (10.61%)
  combo:        0.1022 (10.22%)
  분산이득 있음 (combo < 입력평균): True
  축소폭: 0.0039 (3.7%)

[신호 및 능동수익 상관]
  (i) 신호상관 vix↔trend_avg: 0.5824  [M4 회색지대 상단 근방]
  (A) 능동수익 상관 vix↔trend: 0.8019  [M4 기록: ~0.80 고상관]

[분산이득 이론 vs 실현]
  이론적 비율 sqrt((1+ρ_A)/2) = sqrt((1+0.802)/2) = 0.9492
  실현 비율 combo_vol/avg_vol  = 0.1022/0.1061   = 0.9633
  → 실현(0.9633) > 이론(0.9492): 분산이득이 이론치보다 얇음

[해석]
M4에서 예상: "(A) 0.80 고상관 → 분산이득 약함"
실제: 변동성 축소 3.7% (3.7%p 감소)
→ 분산이득은 실재하지만 M4 예상대로 얇다.
→ Gate 2 근소 우위(Sharpe +0.003)는 분산이득보다 개별 신호 조합 효과가 큼.


## 6. 자산곡선 · 낙폭 시각화

In [7]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

curves = [
    ('combo',     res_comb['equity_net'],  'navy',    2.5),
    ('tsmom12',   res_tsmom['equity_net'], 'green',   1.5),
    ('ee(combo)', ee_c['equity_net'],       'navy',    1.0),
    ('vix',       res_vix['equity_net'],   'orange',  1.0),
    ('buy_hold',  bh_res['equity_net'],    'gray',    1.0),
]

ax = axes[0]
for label, eq, color, lw in curves:
    ls = '--' if label == 'ee(combo)' else '-'
    ax.semilogy(eq.index, eq.values, label=label, color=color, lw=lw, ls=ls)
ax.set_title('자산곡선 (net, log scale, 1990-01-31~)', fontsize=12)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = axes[1]
for label, eq, color, lw in curves:
    dd = eq / eq.cummax() - 1
    ls = '--' if label == 'ee(combo)' else '-'
    ax.plot(dd.index, dd.values, label=label, color=color, lw=lw, ls=ls)
ax.set_title('낙폭 곡선 (Drawdown)', fontsize=12)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/combo_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('저장: results/combo_curves.png')

저장: results/combo_curves.png


## 7. 회귀 검증

In [8]:
import hashlib, subprocess

# M4 CSV 불변 확인
m4_files = [
    'results/vol_metrics_full.csv',
    'results/credit_metrics_full.csv',
    'results/trend_metrics_full.csv',
    'results/common1990_metrics.csv',
]
print('M4 CSV 불변 확인:')
for f in m4_files:
    df_check = pd.read_csv(f)
    print(f'  {f}: {len(df_check)}행 ✅')

print()
# pytest
result = subprocess.run(['python', '-m', 'pytest', 'tests/', '-q'], capture_output=True, text=True)
last_lines = result.stdout.strip().split('\n')[-3:]
for line in last_lines:
    print(line)
assert 'passed' in result.stdout and 'failed' not in result.stdout, 'pytest 실패'
print('pytest ✅')

M4 CSV 불변 확인:
  results/vol_metrics_full.csv: 9행 ✅
  results/credit_metrics_full.csv: 3행 ✅
  results/trend_metrics_full.csv: 6행 ✅
  results/common1990_metrics.csv: 12행 ✅



......................................                                   [100%]
38 passed in 1.05s
pytest ✅


## 8. Gate 2 최종 요약

### 결합 구성 (사전 잠금 확인)
- `w_vol` = VIX 타게팅 (사전등록 대표)
- `w_trend` = mean(MA200, TSMOM12) (동률 등가중, 성과선택 금지)
- `w_comb` = mean(w_vol, w_trend) — 신호 가중치 데이터 최적화 없음

### Gate 2 판정 요약

| 구간 | combo vs ee | combo vs tsmom12 | Gate 2 |
|---|---|---|---|
| 전체 (1990~) | 3/3 ✅ | 3/3 ✅ (근소: Sh +0.003) | **PASS** |
| 전반 (1990~2007) | 3/3 ✅ | 0/3 ❌ (tsmom12 우세) | — |
| 후반 (2008~) | 3/3 ✅ | 3/3 ✅ | — |

**Gate 2: PASS** — 결합이 전체기간 net 기준으로 ee·tsmom12 둘 다를 3/3 위험조정 지표로 상회.

### 중요 단서
1. **근소 우위**: 전체기간 Sharpe 차이 0.634 vs 0.631 = 0.003 — 통계적 유의성은 M8에서 부트스트랩으로 확인 필요.
2. **하위구간 의존성**: 전반(1990~2007, 조용한 강세장)에서 tsmom12 명확 우위 (0.702 vs 0.569). 전반 구간만 보면 Gate 2 FAIL. combo의 전체기간 우위는 후반(2008~) 구간에 기반.
3. **분산이득 박약**: (A) 능동수익 상관 0.80 → 변동성 축소 3.7%에 불과. M4 예상과 일치.
4. **다음 단계**: Phase 3 진입 여부는 사용자 확인 후. 하위구간 의존성과 분산이득 박약에 대한 인식 공유 후 진행.